## RR desde PPG con tres modulaciones y fusión con abstención
- **RIIV** (línea base: media de pico y valle), **RIAV** (amplitud pico − valle) y **RIFV** (intervalo entre latidos), una muestra por latido.
- RR de cada una con la misma FFT (banda 6–35 rpm), y fusión tipo Smart Fusion (Karlen et al., 2013):
  si las tres coinciden (desviación estándar ≤ umbral) se promedian; si no, la ventana **no se estima**.
- **Desarrollo: BIDMC** (se elige el umbral con una regla fijada de antemano y se congela).
  **Prueba externa: CapnoBase**, sin tocar nada.
- Comparación con la FFT de una sola modulación ya validada (notebooks 02 y 03).

In [ ]:
# f-1 — Imports, configuración y rutas
import json
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, decimate, detrend, filtfilt, find_peaks, resample_poly

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE_DIR / "results"
CFG = json.loads((OUT / "config_congelada.json").read_text())
FS = 125
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MIN, RPM_MAX = CFG["rpm_min"], CFG["rpm_max"]
N = VENTANA_S * FS
FS_MOD = 4                               # Hz: remuestreo de las series latido a latido
MIN_LATIDOS = 15                         # por ventana de 30 s (≈ 30 lpm); si hay menos, no se estima

BIDMC = [f"bidmc{i:02d}" for i in range(1, 54)]
CAPNO_DIR = BASE_DIR / "data" / "capnobase" / "csv"
CAPNO = sorted(p.name.replace("_signal.csv", "") for p in CAPNO_DIR.glob("*_signal.csv"))

# Regla de selección del umbral, fijada ANTES de ver resultados:
# el umbral (rpm) con menor MAE mediano por sujeto en BIDMC entre los que estiman ≥ 70 % de las ventanas.
UMBRALES = [1, 2, 3, 4, 5, 6, np.inf]
COBERTURA_MIN = 0.70
print(len(BIDMC), "BIDMC |", len(CAPNO), "CapnoBase")


In [ ]:
# f-2 — Lectura de registros (igual que en 04)
def leer_bidmc(p):
    ruta = str(BASE_DIR / "data" / p)
    rec = wfdb.rdrecord(ruta)
    ppg = rec.p_signal[:, [s.strip(" ,").upper() for s in rec.sig_name].index("PLETH")]
    ann = wfdb.rdann(ruta, "breath")
    aux = np.array([(a or "").strip(" \x00") for a in ann.aux_note])
    fa = ann.fs or rec.fs
    return ppg, {k: np.sort(ann.sample[aux == k]) / fa for k in ("ann1", "ann2")}

def _lista(c):
    return np.array([]) if pd.isna(c) or str(c).strip() == "" else np.array(str(c).split(), dtype=float)

def leer_capno(c):
    sig = pd.read_csv(CAPNO_DIR / f"{c}_signal.csv")
    par = pd.read_csv(CAPNO_DIR / f"{c}_param.csv").iloc[0]
    lab = pd.read_csv(CAPNO_DIR / f"{c}_labels.csv", keep_default_na=False).iloc[0]
    ppg = resample_poly(sig["pleth_y"].to_numpy(float), FS, int(par["samplingrate_pleth"]))
    return ppg, {"ann1": np.sort(_lista(lab["co2_startexp_x"])) / int(par["samplingrate_co2"])}


In [ ]:
# f-3 — Funciones: latidos, modulaciones, FFT y referencia
def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def pico_banda(x, fs, nfft):
    """RR (rpm) del pico espectral en la banda, tras destendencia y Hann."""
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, 1 / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    banda = (f >= RPM_MIN) & (f <= RPM_MAX)
    return f[np.argmax(np.where(banda, P, -np.inf))]

def rr_fft_riiv(seg):
    """FFT de una modulación ya validada (notebooks 02/03): PPG filtrada → 5 Hz → FFT."""
    q = int(FS // 5)
    x = decimate(decimate(seg, 5, zero_phase=True), q // 5, zero_phase=True)
    return pico_banda(x, FS / q, 2048)

def detectar_latidos(ppg):
    """Picos sistólicos y valles previos en la PPG filtrada 0.5–8 Hz.
    Separación mínima 0.33 s (180 lpm); prominencia relativa a la mediana robusta del registro."""
    b, a = butter(2, [0.5, 8], btype="band", fs=FS)
    x = filtfilt(b, a, ppg)
    escala = 1.4826 * np.median(np.abs(x - np.median(x)))
    picos, _ = find_peaks(x, distance=int(0.33 * FS), prominence=0.5 * escala)
    valles = np.array([p0 + np.argmin(x[p0:p1]) for p0, p1 in zip(picos[:-1], picos[1:])], int)
    return picos[1:], valles            # cada pico con el valle que lo precede

def series_modulacion(ppg, picos, valles):
    """Series latido a latido: tiempo (s) del pico, RIIV, RIAV, RIFV.
    - Tiempo del pico refinado por interpolación parabólica (la cuantización de 8 ms a 125 Hz
      es comparable a la propia arritmia sinusal respiratoria).
    - Se descartan latidos cuyo intervalo se aleja > 30 % de la mediana local (11 latidos):
      un latido perdido o falso crea un escalón que domina el espectro respiratorio."""
    y = ppg - np.median(ppg)
    a, b, c = y[picos - 1], y[picos], y[picos + 1]
    den = a - 2 * b + c
    delta = np.where(den != 0, 0.5 * (a - c) / np.where(den != 0, den, 1), 0.0)
    t = (picos + np.clip(delta, -0.5, 0.5)) / FS
    ibi = np.diff(t)
    t, picos, valles = t[1:], picos[1:], valles[1:]
    med = pd.Series(ibi).rolling(11, center=True, min_periods=3).median().to_numpy()
    ok = (ibi >= 0.33) & (ibi <= 1.5) & (np.abs(ibi - med) <= 0.3 * med)
    riiv = (y[picos] + y[valles]) / 2          # línea base = media de pico y valle (Charlton et al.)
    return t[ok], riiv[ok], (y[picos] - y[valles])[ok], ibi[ok]

def rr_modulacion(t, v, t0):
    """Interpola la serie latido a latido a 4 Hz dentro de la ventana y estima la RR."""
    m = (t >= t0) & (t < t0 + VENTANA_S)
    if m.sum() < MIN_LATIDOS:
        return np.nan
    tg = t0 + np.arange(VENTANA_S * FS_MOD) / FS_MOD
    return pico_banda(np.interp(tg, t[m], v[m]), FS_MOD, 1024)


In [ ]:
# f-4 — Cálculo por ventana en ambas bases
filas = []
for base, ids, lector in [("BIDMC", BIDMC, leer_bidmc), ("CapnoBase", CAPNO, leer_capno)]:
    for rid in ids:
        ppg, marcas = lector(rid)
        b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
        riiv_fft = filtfilt(b, a, ppg)
        b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
        riiv_art = filtfilt(b, a, ppg)
        n5 = 5 * FS
        tipico = np.median([np.std(riiv_art[i:i + n5]) for i in range(0, len(ppg) - n5 + 1, n5)])
        t_l, iiv, iav, ifv = series_modulacion(ppg, *detectar_latidos(ppg))
        for t0 in np.round(np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S), 3):
            i0 = int(round(t0 * FS))
            exc = max(np.std(riiv_art[i:i + n5]) for i in range(i0, i0 + N - n5 + 1, n5)) / tipico
            r1 = rr_ref(marcas["ann1"], t0, t0 + VENTANA_S)
            fiable = True if "ann2" not in marcas else \
                abs(r1 - rr_ref(marcas["ann2"], t0, t0 + VENTANA_S)) <= CFG["umbral_anot"]
            filas.append({"Base": base, "Sujeto": rid, "Inicio (s)": t0,
                          "RR referencia": r1,
                          "Válida": bool(exc <= CFG["umbral_art"] and np.isfinite(r1) and fiable),
                          "RR FFT 1 mod": rr_fft_riiv(riiv_fft[i0:i0 + N]),
                          "RR RIIV": rr_modulacion(t_l, iiv, t0),
                          "RR RIAV": rr_modulacion(t_l, iav, t0),
                          "RR RIFV": rr_modulacion(t_l, ifv, t0)})
D = pd.DataFrame(filas)
mods = D[["RR RIIV", "RR RIAV", "RR RIFV"]]
D["Media 3 mod"] = mods.mean(axis=1, skipna=False)
D["Desv 3 mod"] = mods.std(axis=1, ddof=0, skipna=False)
D.to_csv(OUT / "fusion_ventanas.csv", index=False)
print(D.groupby("Base")["Válida"].agg(["size", "sum"]))


In [ ]:
# f-5 — Desarrollo en BIDMC: elegir el umbral de concordancia con la regla fijada
def evaluar(d, col, umbral=None):
    """MAE mediano por sujeto y cobertura (fracción de ventanas válidas con estimación)."""
    est = d[col].notna()
    if umbral is not None:
        est &= d["Desv 3 mod"] <= umbral
    e = (d.loc[est, col] - d.loc[est, "RR referencia"]).abs()
    por_suj = e.groupby(d.loc[est, "Sujeto"]).mean()
    return {"MAE mediano": por_suj.median(), "MAE medio": e.mean(),
            "% ≤ 2 rpm": 100 * (e <= 2).mean(), "Cobertura %": 100 * est.mean()}

VB = D[(D["Base"] == "BIDMC") & D["Válida"]]
filas = [{"Umbral (rpm)": u, **evaluar(VB, "Media 3 mod", u)} for u in UMBRALES]
sel = pd.DataFrame(filas)
print(sel.round(2).to_string(index=False))
cand = sel[sel["Cobertura %"] >= 100 * COBERTURA_MIN]
UMBRAL = float(cand.loc[cand["MAE mediano"].idxmin(), "Umbral (rpm)"])
print(f"\nUmbral elegido en BIDMC: {UMBRAL} rpm (congelado para CapnoBase)")
CFG_F = {**CFG, "fusion_umbral_rpm": UMBRAL, "fusion_fs_mod": FS_MOD, "fusion_min_latidos": MIN_LATIDOS,
         "fusion_diseño": "umbral elegido en BIDMC (MAE mediano mínimo con cobertura >= 70 %)"}
(OUT / "config_fusion.json").write_text(json.dumps(CFG_F, indent=2, default=float))


In [ ]:
# f-6 — Resultados: BIDMC (desarrollo) y CapnoBase (prueba externa)
filas = []
for base, g in D[D["Válida"]].groupby("Base"):
    for nombre, col, u in [("FFT 1 modulación (validada)", "RR FFT 1 mod", None),
                           ("RIIV latido a latido", "RR RIIV", None),
                           ("RIAV", "RR RIAV", None), ("RIFV", "RR RIFV", None),
                           ("Media 3 mod, sin abstención", "Media 3 mod", None),
                           (f"Fusión con abstención ({UMBRAL:g} rpm)", "Media 3 mod", UMBRAL)]:
        filas.append({"Base": base, "Método": nombre, **evaluar(g, col, u)})
R = pd.DataFrame(filas)
R.to_csv(OUT / "fusion_resumen.csv", index=False)
print(R.round(2).to_string(index=False))
